# Lightweight spatiotemporal feature combinations — XGBoost

**Run this only to screen combinations before any additional full-history model fits.**

Each configuration uses the same 10,000 historical training rows, up to 1,000 August 2025 held-out homes, and 220 tuned XGBoost trees. We keep the existing 47-word binary BoW baseline unchanged.

- **Feature combinations:** spline + geo context, spline + property-type quota, spline + market, spline + both geo context and type quota.
- **Residual combinations:** spline + tensor GAM and spline+geo + tensor GAM. Crucially, each GAM is fitted on *genuinely out-of-time residuals from that exact feature model* in earlier months (not the ordinary XGBoost residuals).

These numbers are a **development screen**, not comparable in absolute magnitude to the full 700-tree baseline. Run all cells, inspect the summary and failure diagnostics, and promote promising variants for subsequent three-month confirmation before full-data evaluation. No original notebooks or saved models are modified by the combination run.


In [ ]:
from IPython.display import display
from spacetime_combination_screen import run_combination_screen, COMBINATIONS, CORRECTIONS

MONTH = "2025-08"
TRAIN_START = "2022-01-01"
TRAIN_CAP = 10000
VALIDATION_CAP = 1000
TREES = 220
CALIBRATION_FOLDS = 2
CALIBRATION_CAP = 300
RESIDUAL_STRENGTH = 0.5
SEED = 42
OUTPUT_DIR = "outputs/spacetime_combination_screen"
COMPARISON_VARIANTS = tuple((*COMBINATIONS, *CORRECTIONS))
print("Comparisons:", COMPARISON_VARIANTS)


In [ ]:
combo_scores, combo_predictions, combo_failures = run_combination_screen(
    validation_month=MONTH, train_start=TRAIN_START,
    train_cap=TRAIN_CAP, validation_cap=VALIDATION_CAP,
    trees=TREES, seed=SEED,
    calibration_folds=CALIBRATION_FOLDS,
    calibration_cap=CALIBRATION_CAP,
    residual_strength=RESIDUAL_STRENGTH,
    combinations=COMPARISON_VARIANTS,
    output_dir=OUTPUT_DIR,
)
display(combo_scores)
if combo_failures:
    display(combo_failures)


In [ ]:
import matplotlib.pyplot as plt
ranking = combo_scores.sort_values("delta_mdape_pp")
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(ranking.variant, ranking.delta_mdape_pp)
ax.axvline(0, color="gray", linewidth=1)
ax.invert_yaxis()
ax.set_xlabel("MdAPE change vs original XGBoost (pp; negative is better)")
ax.set_title("Quick feature / GAM combination screen")
fig.tight_layout()
plt.show()
print(f"Saved summary, predictions, failures and settings in {OUTPUT_DIR}")


### Interpreting the results

A combination must beat **spline-only**, not just the original baseline, to justify additional computational cost. Prefer gains in both MAPE and MdAPE across more than one development month, and avoid promoting variants based on tiny improvements from this single holdout.

We should run a larger three-month follow-up for any promising **pure feature** combination, then only run it with all historical sales and the 700-tree XGBoost configuration. For GAM combinations, full-strength calibration must also be recomputed for the combined model's own out-of-time residuals; **do not reuse the tensor-GAM calibration from original XGBoost**.
